# 05 — RAG Evaluation and Context Experiments

This notebook evaluates four controlled configurations using the same 10 supported and 3 unsupported questions. Only prompt style and requested context size change:

| Configuration | Prompt | Requested context |
|---|---|---:|
| `basic_top3` | Simple context instruction | Top-3 |
| `basic_top5` | Simple context instruction | Top-5 |
| `grounded_top3` | Evidence-only instruction + gate | Top-3 |
| `grounded_top5` | Evidence-only instruction + gate | Top-5 |

The generator and 128-token retrieval index remain constant. This isolates the two design variables instead of changing several components simultaneously.

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

results = json.loads(
    (PROJECT_ROOT / 'evaluation/rag_experiment_results.json').read_text(encoding='utf-8')
)
rows = pd.DataFrame(results['rows'])
summary = pd.DataFrame(results['summary'])
print('Questions:', results['question_count'])
print('Generated/evaluated runs:', len(rows))
print('Initial evidence threshold:', results['minimum_evidence_score'])

Questions: 13
Generated/evaluated runs: 52
Initial evidence threshold: 0.45


## Checkpoint 1 — What each metric means

- **Mean answer-point coverage:** expected facts detected in generated answers using transparent accepted-phrase rules.
- **All-points rate:** supported questions for which every expected point was detected.
- **Expected-source rate:** at least one manually accepted document/page reached the generator prompt.
- **Unsupported refusal rate:** unsupported questions answered with explicit refusal wording.
- **Evidence-gate rate:** unsupported questions stopped before the LLM was called.

Phrase-based answer scoring is reproducible but imperfect: paraphrases can be missed, and matching a phrase does not prove faithfulness. Manual failure review remains necessary.

In [2]:
display_columns = [
    'configuration', 'mean_answer_point_coverage', 'all_answer_points_rate',
    'expected_source_rate', 'unsupported_refusal_rate',
    'evidence_gate_rate_unsupported', 'mean_input_tokens_when_called',
    'mean_latency_when_called', 'total_generation_seconds'
]
summary[display_columns].style.format({
    'mean_answer_point_coverage': '{:.1%}',
    'all_answer_points_rate': '{:.1%}',
    'expected_source_rate': '{:.1%}',
    'unsupported_refusal_rate': '{:.1%}',
    'evidence_gate_rate_unsupported': '{:.1%}',
    'mean_input_tokens_when_called': '{:.1f}',
    'mean_latency_when_called': '{:.2f}',
    'total_generation_seconds': '{:.1f}'
})

,configuration,mean_answer_point_coverage,all_answer_points_rate,expected_source_rate,unsupported_refusal_rate,evidence_gate_rate_unsupported,mean_input_tokens_when_called,mean_latency_when_called,total_generation_seconds
0,basic_top3,55.0%,40.0%,90.0%,66.7%,0.0%,443.2,11.85,154.0
1,basic_top5,55.0%,40.0%,90.0%,66.7%,0.0%,683.2,12.71,165.2
2,grounded_top3,58.3%,40.0%,90.0%,66.7%,66.7%,476.5,8.12,89.3
3,grounded_top5,55.0%,30.0%,90.0%,66.7%,66.7%,718.8,8.03,88.3


## Checkpoint 2 — Top-5 added context but did not improve quality

For the grounded prompt, Top-5 reduced mean answer-point coverage and all-points rate while substantially increasing prompt size. More context can introduce repetition or unrelated detail; it is not automatically better.

In [3]:
g3 = summary.set_index('configuration').loc['grounded_top3']
g5 = summary.set_index('configuration').loc['grounded_top5']
print('Grounded Top-3 answer-point coverage:', f"{g3['mean_answer_point_coverage']:.1%}")
print('Grounded Top-5 answer-point coverage:', f"{g5['mean_answer_point_coverage']:.1%}")
print('Grounded Top-3 all-points rate:', f"{g3['all_answer_points_rate']:.1%}")
print('Grounded Top-5 all-points rate:', f"{g5['all_answer_points_rate']:.1%}")
print('Top-5 prompt-token increase:', f"{(g5['mean_input_tokens_when_called']/g3['mean_input_tokens_when_called']-1):.1%}")
print('Expected-source rate remained:', f"{g3['expected_source_rate']:.1%}")

Grounded Top-3 answer-point coverage: 58.3%
Grounded Top-5 answer-point coverage: 55.0%
Grounded Top-3 all-points rate: 40.0%
Grounded Top-5 all-points rate: 30.0%
Top-5 prompt-token increase: 50.9%
Expected-source rate remained: 90.0%


## Checkpoint 3 — Inspect failures, not only averages

The initial `0.45` gate stopped two unsupported questions, but U03 scored `0.510` and reached the LLM. A grounded prompt is an instruction, not a security boundary: Qwen then invented a Sandisk equipment answer. The basic prompt also invented operating temperatures for U02.

In [4]:
failure_ids = [
    ('basic_top3', 'U02'),
    ('grounded_top3', 'U03'),
    ('grounded_top3', 'Q04')
]
for configuration, question_id in failure_ids:
    row = rows[(rows.configuration == configuration) & (rows.question_id == question_id)].iloc[0]
    print('\n', '=' * 100)
    print(configuration, question_id, '| top similarity:', round(row.top_similarity, 3))
    print('Expected source reached prompt:', row.expected_source_present)
    print('Gate triggered:', row.evidence_gate_triggered)
    print('Answer:', row.answer[:1200])


basic_top3 U02 | top similarity: 0.359
Expected source reached prompt: None
Gate triggered: False
Answer: According to the provided context, Sandisk's SMT placement machine should be operated at an operating temperature of around 50°C (122°F). This information is directly stated in the first sentence of the given evidence:

"SANDISK SMT Placement Machine Operating Temperature Range: 20°C to 70°C (68°F to 158°F)"

This range indicates that Sandisk recommends operating the SMT placement machine within this temperature range to ensure

Retrieved sources:
1. NIST.AMS.100-2 — PDF page 48
2. NIST.AMS.100-2 — PDF page 34
3. NIST.AMS.100-2 — PDF page 1

grounded_top3 U03 | top similarity: 0.51
Expected source reached prompt: None
Gate triggered: False
Answer: Based on the provided evidence, the Sandisk factory equipment model with the highest failure rate is the 8TB HDD model. The evidence states that "Failure rates vary significantly between different types of equipment," indicating that the

## Checkpoint 4 — Calibrate the evidence threshold

Cosine similarity is not a probability. Threshold selection compares score distributions. On this small evaluation set, the lowest supported top score is above the highest unsupported score, so their midpoint is a possible provisional boundary. Because the same three unsupported questions are used for calibration and reporting, this is optimistic and requires future validation on new questions.

In [5]:
score_rows = rows[rows.configuration == 'grounded_top3'][
    ['question_id', 'supported', 'top_similarity']
].drop_duplicates()
minimum_supported = score_rows[score_rows.supported].top_similarity.min()
maximum_unsupported = score_rows[~score_rows.supported].top_similarity.max()
provisional_threshold = (minimum_supported + maximum_unsupported) / 2
print('Minimum supported score:', round(minimum_supported, 3))
print('Maximum unsupported score:', round(maximum_unsupported, 3))
print('Midpoint provisional threshold:', round(provisional_threshold, 3))
score_rows.sort_values('top_similarity').style.format({'top_similarity': '{:.3f}'})

Minimum supported score: 0.657
Maximum unsupported score: 0.51
Midpoint provisional threshold: 0.584


,question_id,supported,top_similarity
36,U01,False,0.336
37,U02,False,0.359
38,U03,False,0.510
29,Q04,True,0.657
27,Q02,True,0.677
34,Q09,True,0.678
33,Q08,True,0.738
35,Q10,True,0.755
30,Q05,True,0.756
32,Q07,True,0.774


In [6]:
unsupported = score_rows[~score_rows.supported].copy()
unsupported['rejected_at_045'] = unsupported.top_similarity < 0.45
unsupported['rejected_at_provisional'] = unsupported.top_similarity < provisional_threshold
print('Unsupported gate rate at 0.45:', f"{unsupported.rejected_at_045.mean():.1%}")
print('Unsupported gate rate at provisional threshold:', f"{unsupported.rejected_at_provisional.mean():.1%}")
unsupported.style.format({'top_similarity': '{:.3f}'})

Unsupported gate rate at 0.45: 66.7%
Unsupported gate rate at provisional threshold: 100.0%


,question_id,supported,top_similarity,rejected_at_045,rejected_at_provisional
36,U01,False,0.336,True,True
37,U02,False,0.359,True,True
38,U03,False,0.510,False,True


## Checkpoint 5 — Sources are deterministic application output

The LLM generates only the answer text. The application appends every unique document/page whose complete chunk was actually placed in the prompt:

```text
Generated answer...

Retrieved sources:
1. NIST.AMS.100-34 — PDF page 15
2. NIST.AMS.100-18 — PDF page 10
```

This supports multi-source answers and prevents invented page numbers. It means ‘sources available to the model’, not that every source supports every sentence.

## Engineering decision

Select **grounded + Top-3** for the MVP. It achieved the strongest measured mean answer-point coverage, matched Top-5 expected-source coverage, used substantially fewer input tokens, and supports pre-generation rejection. Replace the initial `0.45` gate with the provisional midpoint (approximately `0.58`), while documenting that this threshold is calibrated on only three unsupported questions.

The principal remaining quality issue is retrieval failure: Q04's accepted evidence page did not reach either Top-3 or Top-5. Prompt engineering cannot recover evidence the retriever failed to supply. Future improvement should target hybrid retrieval or query expansion before using a larger generator.